# 🗺 Zone Clustering — Peddapalli District Risk Mapping

Clusters geographic accident data into risk zones using **K-Means** on (lat, lon, risk_score).

Outputs:
- Cluster labels per row
- Per-cluster risk profiles
- Static visualisation of Peddapalli accident zones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.style.use('dark_background')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

print("Libraries loaded ✅")

## 1. Load Data

In [ ]:
df = pd.read_csv('../backend/data/peddapalli_accidents.csv')
acc = df[df['accident_occurred'] == 1].copy()
print(f"Accident records: {len(acc)}")
acc[['latitude','longitude','risk_score']].describe()

## 2. Elbow Method — Optimal K

In [ ]:
X_geo = acc[['latitude','longitude','risk_score']].values
scaler_geo = StandardScaler()
X_geo_s    = scaler_geo.fit_transform(X_geo)

inertias   = []
sil_scores = []
K_range    = range(2, 12)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_geo_s)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_geo_s, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_range, inertias, marker='o', color='#FF3B30', lw=2.5, markersize=7)
axes[0].set_title('Elbow Method — Inertia', fontsize=13)
axes[0].set_xlabel('K (clusters)'); axes[0].set_ylabel('Inertia')
axes[0].axvline(x=8, color='#FFCC00', linestyle='--', alpha=0.7, label='K=8 chosen')
axes[0].legend()

axes[1].plot(K_range, sil_scores, marker='s', color='#34C759', lw=2.5, markersize=7)
axes[1].set_title('Silhouette Score', fontsize=13)
axes[1].set_xlabel('K (clusters)'); axes[1].set_ylabel('Score')
axes[1].axvline(x=8, color='#FFCC00', linestyle='--', alpha=0.7, label='K=8 chosen')
axes[1].legend()

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Best silhouette at K={list(K_range)[np.argmax(sil_scores)]}: {max(sil_scores):.4f}")

## 3. Final Clustering (K=8)

In [ ]:
K = 8
km_final = KMeans(n_clusters=K, random_state=42, n_init=20)
acc['cluster'] = km_final.fit_predict(X_geo_s)

print(f"Cluster distribution:")
print(acc['cluster'].value_counts().sort_index())

## 4. Cluster Risk Profiles

In [ ]:
profile = acc.groupby('cluster').agg(
    n_accidents     = ('accident_id', 'count'),
    avg_risk        = ('risk_score',  'mean'),
    fatal_count     = ('severity',    lambda x: (x=='Fatal').sum()),
    serious_count   = ('severity',    lambda x: (x=='Serious').sum()),
    minor_count     = ('severity',    lambda x: (x=='Minor').sum()),
    center_lat      = ('latitude',    'mean'),
    center_lon      = ('longitude',   'mean'),
    top_weather     = ('weather',     lambda x: x.mode()[0]),
    top_road_type   = ('road_type',   lambda x: x.mode()[0]),
).sort_values('avg_risk', ascending=False)

def risk_label(score):
    if score >= 70: return 'Critical'
    if score >= 50: return 'High'
    if score >= 30: return 'Medium'
    return 'Low'

profile['risk_label'] = profile['avg_risk'].apply(risk_label)
print(profile[['n_accidents','avg_risk','fatal_count','risk_label','top_weather','top_road_type']])

## 5. Spatial Visualisation

In [ ]:
RISK_COLORS = {
    'Critical': '#FF3B30',
    'High':     '#FF9500',
    'Medium':   '#FFCC00',
    'Low':      '#34C759',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: scatter coloured by cluster
scatter_colors = [RISK_COLORS[risk_label(acc[acc['cluster']==c]['risk_score'].mean())]
                  for c in acc['cluster']]

axes[0].scatter(acc['longitude'], acc['latitude'],
                c=scatter_colors, s=18, alpha=0.6, edgecolors='none')

# Cluster centroids
centers_geo = scaler_geo.inverse_transform(km_final.cluster_centers_)
for i, (lat, lon, risk) in enumerate(centers_geo):
    color = RISK_COLORS[risk_label(risk)]
    axes[0].scatter(lon, lat, s=220, c=color, edgecolors='white',
                    linewidths=1.5, zorder=5, marker='*')
    axes[0].annotate(f'Z{i}', (lon, lat), fontsize=8,
                     ha='center', va='bottom', color='white',
                     xytext=(0,10), textcoords='offset points')

axes[0].set_xlabel('Longitude', fontsize=11)
axes[0].set_ylabel('Latitude', fontsize=11)
axes[0].set_title('Peddapalli District — Accident Clusters', fontsize=13, pad=12)

legend_handles = [mpatches.Patch(facecolor=c, label=l)
                  for l, c in RISK_COLORS.items()]
axes[0].legend(handles=legend_handles, loc='lower right', fontsize=9)

# Right: risk profile bar chart
prof_sorted = profile.sort_values('avg_risk', ascending=True)
bar_colors  = [RISK_COLORS[l] for l in prof_sorted['risk_label']]
axes[1].barh([f'Zone {i}' for i in prof_sorted.index],
             prof_sorted['avg_risk'],
             color=bar_colors, alpha=0.9, edgecolor='none')
axes[1].set_xlabel('Average Risk Score', fontsize=11)
axes[1].set_title('Risk Score by Cluster', fontsize=13, pad=12)
axes[1].axvline(x=50, color='white', linestyle='--', alpha=0.3)
axes[1].axvline(x=70, color='#FF3B30', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('zone_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Risk Hotspot Heatmap

In [ ]:
# Risk score heatmap over the district lat/lon grid
from scipy.stats import binned_statistic_2d

lat_bins = np.linspace(acc['latitude'].min()-0.01,  acc['latitude'].max()+0.01,  40)
lon_bins = np.linspace(acc['longitude'].min()-0.01, acc['longitude'].max()+0.01, 40)

grid, _, _, _ = binned_statistic_2d(
    acc['longitude'], acc['latitude'], acc['risk_score'],
    statistic='mean', bins=[lon_bins, lat_bins]
)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(grid.T, origin='lower', aspect='auto',
               extent=[lon_bins[0], lon_bins[-1], lat_bins[0], lat_bins[-1]],
               cmap='RdYlGn_r', alpha=0.85, vmin=0, vmax=100)
ax.scatter(acc['longitude'], acc['latitude'],
           s=8, c='white', alpha=0.25, edgecolors='none', label='Accidents')
plt.colorbar(im, ax=ax, shrink=0.7, label='Avg Risk Score')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Risk Score Heatmap — Peddapalli District', fontsize=14, pad=14)
plt.tight_layout()
plt.savefig('risk_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Export Zone Profiles

In [ ]:
profile.to_csv('zone_profiles.csv')
print("Zone profiles saved to zone_profiles.csv ✅")
print()
print("=== Summary ===")
print(f"Total clusters: {K}")
print(f"Critical zones: {(profile['risk_label']=='Critical').sum()}")
print(f"High zones:     {(profile['risk_label']=='High').sum()}")
print(f"Medium zones:   {(profile['risk_label']=='Medium').sum()}")
print(f"Low zones:      {(profile['risk_label']=='Low').sum()}")